# Data on the Line
*Turning Demand Insights into Balanced Production* · Vaka çalışması · KoçDigital analitik ekibi · 120 dakika · Tahmin tarihi: **31 Aralık 2025**

| Dakika | Ne yapıyoruz? | Notebook'ta |
|---|---|---|
| 0–10 | Vaka metnini okuyoruz | — |
| 10–45 | Talebi tahmin ediyoruz | Bölüm 1 |
| 45 | Tahmin dosyasını yüklüyoruz; birinci şifre duyuruluyor | Adım 1.3'te inen `TeamXX_forecast.csv` → “45. dakika tahmini” formu |
| 45–57 | Üretim planını ve takt süresini hesaplıyoruz | Bölüm 2 |
| 57–90 | Hattı dengeliyoruz; jüri bu sırada uğrar | Bölüm 3 |
| 90–95 | Hat tasarımının ve hat slaytının tamamlanması | Bölüm 5'teki cevaplar, tahmin ve plan slaytı, hat slaytı |
| 95 | İkinci şifre duyuruluyor | Bölüm 4 |
| 95–115 | ACİL siparişi değerlendiriyor, ACİL slaytını bitiriyoruz | Bölüm 4 ve Bölüm 5 |
| 115 | Final teslim | Üç dosya → “Final teslim” formu |
| 115–120 | Kısa kapanış | — |

Oturum toplam 120 dakika sürüyor ve 115. dakikadan sonra dosya kabul edilmiyor. Bu yüzden slaytları en sona bırakmayın: Bölüm 2'den (üretim planı) itibaren bir kişi slaytları analizle eş zamanlı doldursun (tahmin ve plan slaytı 57. dakikaya, hat slaytı 95. dakikaya kadar hazır olsun); 95. dakikadan sonra yalnızca ACİL slaytı ve son düzeltmeler kalsın. Dosyaları hazır olur olmaz yükleyin; yeniden yüklerseniz son yükleme sayılır.

**Notebook nasıl ilerliyor?**

- Her bölüm kısa bir açıklamayla başlıyor, ardından adım adım ilerliyor.
- Açıklamanın hemen ardından **HAZIR** başlıklı bir kod hücresi var: veriyi yükleyen, hataları ölçen, sonuçlarınızı kontrol eden ve teslim dosyalarını yazan fonksiyonlar. Bu hücreleri çalıştırmanız yeterli; merak ederseniz okuyabilirsiniz, ama değiştirmeniz gerekmiyor.
- `# TODO` ile işaretli hücreler sizin.
- Her bölümün sonunda, doğru yolda olup olmadığınızı gösteren bir kontrol hücresi var. Ocak, çözümlü örnek aydır: kontrol hücresi ocak için doğru değerleri gösterir. Diğer aylar ve tasarım kararlarınız ise size kalıyor.
- Kurallar vaka metnindeki yazışmalarda. Hangi kuralın ne anlama geldiğini siz çıkaracaksınız.
- Yapay zekâ asistanlarından yararlanabilirsiniz; ama jüri, teslim ettiğiniz her satırı açıklamanızı isteyebilir.

İş bölümü için bir öneri: bir kişi tahmine, bir kişi plana ve hat dengelemeye, bir kişi de baştan itibaren slaytlara odaklansın. Hat modelini kurmak için takt süresini beklemeniz gerekmez; `C, MIX = 85, {"ND100": 0.4, "ND200": 0.35, "ND300": 0.25}` gibi geçici değerlerle erkenden denemeye başlayabilirsiniz.

### Başlarken

**Önce takım numaranızı yazın.** Aşağıdaki ilk hücredeki `TEAM = "Team__"` satırına eğitmeninizin verdiği numarayı yazın, örneğin `TEAM = "Team03"`. Teslim dosyalarınızın adları bu numarayla başlar (`Team03_forecast.csv`, `Team03_submission.zip`); dosyaları iki forma yükleyeceksiniz: “45. dakika tahmini” ve “Final teslim” (Google'daki adları *Data on the Line – Minute-45 forecast* ve *Data on the Line – Final submission*). Formlara Colab'de kullandığınız Google hesabıyla girin.

Notebook'u Colab'de açtıysanız sonraki hücre veri dosyalarını GitHub'dan otomatik olarak indirir; sizin hiçbir dosya yüklemeniz gerekmez. Hücreleri çalıştırdığınızda **“Veri yüklendi ✓”** yazısını görmelisiniz.

Colab, oturum uzun süre boşta kalırsa çalışma dosyalarını siler. Yaptıklarınızı kaybetmemek için ara ara *Dosya → Drive'a kopya kaydet* ile notebook'unuzu kaydedebilirsiniz.

Kendi bilgisayarınızda Jupyter ile çalışıyorsanız notebook ile `data/` klasörü aynı klasörde bulunmalı.

In [ ]:
TEAM = "Team__"   # eğitmeninizin verdiği takım numarası, örneğin "Team03"

In [ ]:
# HAZIR · Başlangıç: kütüphaneler ve veri

import importlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
import zipfile
import zlib
from datetime import datetime
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Colab'de veri dosyaları GitHub'dan otomatik iner; kendi bilgisayarınızda bu adım atlanır
REPO_URL = "https://github.com/burcu-koca-kd/data-on-the-line.git"
CONTENT = Path("/content")
REPO = CONTENT / "data-on-the-line"
CASE_DIR = REPO / "tr/2_Vaka"


def _clone():
    try:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(REPO)],
                       check=True, capture_output=True)
    except subprocess.CalledProcessError:
        print("✗ Veri dosyaları GitHub'dan indirilemedi. İnternet bağlantınızı kontrol edin, "
              "ardından notebook bağlantısını yeniden açın ya da bu hücreyi tekrar çalıştırın.")
        raise RuntimeError("Veri dosyaları indirilemedi (yukarıdaki mesaja bakın).") from None


def _update():
    """İndirilmiş kopyayı GitHub'daki son sürüme getirir; release/ ve submission/ klasörlerine
    dokunmaz."""
    git = ["git", "-C", str(REPO)]
    fetched = subprocess.run(git + ["fetch", "-q", "--depth", "1"], capture_output=True)
    if fetched.returncode == 0:
        subprocess.run(git + ["reset", "-q", "--hard", "origin/main"], capture_output=True)


def _uploaded_case():
    """Depoda vaka klasörü yoksa Colab'e ya da Drive'a yüklenmiş bir 2_Vaka klasörünü (ya da
    2_Vaka.zip dosyasını) arar."""
    for archive in sorted(CONTENT.glob("2_Vaka*.zip")):
        with zipfile.ZipFile(archive) as z:
            z.extractall(CONTENT)
    for root in (CONTENT, CONTENT / "drive" / "MyDrive"):
        for folder in (root / "2_Vaka", *root.glob("*/2_Vaka"), root):
            if (folder / "data" / "tasks.csv").exists():
                return folder
    return None


if "google.colab" in sys.modules:
    # daha önce indirilmiş bir kopya varsa en son sürüme getirilir (internet yoksa sessizce
    # geçilir)
    if REPO.exists():
        _update()
    # eski kopyada vaka klasörü yoksa depo baştan indirilir
    if not CASE_DIR.exists():
        shutil.rmtree(REPO, ignore_errors=True)
        _clone()
    # vaka klasörü depoda henüz yoksa: yüklenmiş bir kopya aranır
    if not CASE_DIR.exists():
        CASE_DIR = _uploaded_case()
        if CASE_DIR is None:
            print("✗ Vaka dosyaları henüz yayımlanmamış görünüyor: depoda 2_Vaka klasörü yok. "
                  "Eğitmeninize haber verin. Eğitmeniniz 2_Vaka klasörünü dağıttıysa: (1) "
                  "2_Vaka.zip dosyasını Colab'in sol kenarındaki Dosyalar paneline sürükleyin ya "
                  "da (2) 2_Vaka klasörünü Drive'ınıza koyup Dosyalar panelinden Drive'ı "
                  "bağlayın; sonra bu hücreyi yeniden çalıştırın.")
            raise RuntimeError("Vaka klasörü bulunamadı (yukarıdaki mesaja bakın).")
        print(f"Depoda vaka klasörü yok; yüklenen kopya kullanılıyor: {CASE_DIR}")
    os.chdir(CASE_DIR)


def _install(*packages):
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(f"pip install {' '.join(packages)} başarısız oldu. Hata mesajının sonu:")
        print(result.stderr[-1500:])


def _pulp_major():
    try:
        return int(version("pulp").split(".")[0])
    except PackageNotFoundError:
        return 0


def _forget_pulp():
    """PuLP'u bellekten siler; bir sonraki import yeni kurulan paketleri görür."""
    for name in [m for m in sys.modules if m == "pulp" or m.startswith("pulp.")]:
        del sys.modules[name]
    importlib.invalidate_caches()


# Colab'de PuLP ya kurulu değildir ya da 4. sürümü kuruludur; vaka 3. sürümle yazıldı
if not 2 <= _pulp_major() < 4:
    _install("pulp>=2.7,<4")
    _forget_pulp()
if _pulp_major() >= 4:
    print("⚠ PuLP 3 kurulamadı, hâlâ 4. sürüm yüklü. Çekirdeği yeniden başlatıp (Çalışma zamanı → "
          "Oturumu yeniden başlat) bu hücreyi tekrar çalıştırın.")
import pulp

DATA, RELEASE, SUBMISSION = Path("data"), Path("release"), Path("submission")
UPLOAD = Path("upload")
SUBMISSION.mkdir(exist_ok=True)
# forma yüklenecek dosyalar (TeamXX_...) burada toplanır
UPLOAD.mkdir(exist_ok=True)


def _team():
    """TEAM değişkenindeki takım numarasını 'Team03' biçiminde döndürür; yazılmamışsa uyarır ve
    None döndürür."""
    name = str(globals().get("TEAM", "")).strip()
    found = (re.fullmatch(r"0*(\d{1,2})", name)
             or re.search(r"(?i)(?:team|tak[ıi]m)\s*[-_]?\s*0*(\d{1,2})(?!\d)", name))
    if not found or int(found.group(1)) == 0:
        print("⚠ Takım numaranız yazılmamış: notebook'un başındaki TEAM = 'Team__' satırına "
              "numaranızı yazın (örneğin TEAM = 'Team03'), o hücreyi ve bu hücreyi yeniden "
              "çalıştırın. Yoksa dosyalarınızın adı TeamXX ile başlar ve eğitmeniniz onları "
              "takımınızla eşleştiremeyebilir.")
        return None
    return f"Team{int(found.group(1)):02d}"


PRODUCTS = ["ND100", "ND200", "ND300"]
MONTHS = ["2026-01", "2026-02", "2026-03"]

history = pd.read_csv(DATA / "demand_history.csv").fillna({"Calendar_Note": ""})
plan_q1 = pd.read_csv(DATA / "promo_price_plan.csv", dtype={"Promo_Status": str})
plan_q1 = plan_q1.fillna({"Calendar_Note": "", "Promo_Status": ""})
calendar = pd.read_csv(DATA / "production_calendar.csv")
products = pd.read_csv(DATA / "products.csv")
tasks = pd.read_csv(DATA / "tasks.csv")
precedence = pd.read_csv(DATA / "precedence.csv")
current = pd.read_csv(DATA / "current_line.csv")

TASKS = list(tasks.Task)
# TIME[iş][ürün]: saniye cinsinden süre
TIME = {r.Task: {p: getattr(r, f"{p}_sec") for p in PRODUCTS} for r in tasks.itertuples()}
GROUP = dict(zip(tasks.Task, tasks.Process_Group))                 # her işin süreç grubu
# (a, b): a'nın istasyonu b'ninkinden sonra gelemez (aynı istasyon olabilir)
ARCS = list(zip(precedence.Predecessor, precedence.Successor))
CURRENT_LINE = dict(zip(current.Task, current.Station))            # bugünkü hat {iş: istasyon}

# kontrol hücrelerinin kullandığı doğru değerler: ocak ve kıyas yöntemi
NAIVE_WAPE, NAIVE_BIAS = 0.1070, -0.0864
JAN_AVAILABLE, JAN_PLANNED, JAN_TAKT = 1039500, 11180, 92.98


def fmt(x, decimals=0):
    """Sayıyı Türkçe biçimde yazar: 1.039.500 ya da 92,98."""
    text = f"{x:,.{decimals}f}"
    return text.replace(",", " ").replace(".", ",").replace(" ", ".")


def pct(x, decimals=1, signed=False):
    """Oranı yüzde olarak yazar: %10,7; signed=True ile +%10,7 ya da −%8,6."""
    sign = ("+" if x >= 0 else "−") if signed else ("−" if x < 0 else "")
    return f"{sign}%{fmt(abs(x) * 100, decimals)}"


def show(table, decimals=2):
    """Tabloyu Türkçe sayı biçimiyle gösterir: 1.039.500 ve 92,98 (tam sayılar ondalıksız)."""
    whole = [c for c in table.select_dtypes("float") if (table[c].dropna() % 1 == 0).all()]
    try:
        styled = table.style.format(precision=decimals, decimal=",", thousands=".")
        display(styled.format(precision=0, decimal=",", thousands=".", subset=whole))
    except (ImportError, AttributeError):          # biçimli tablolar için jinja2 gerekir
        display(table)


def _as_dict(values):
    """Sözlüğü ya da pandas Series'i düz bir sözlüğe çevirir; None ise None kalır."""
    return None if values is None else dict(values)


def add_features(df):
    """Modellerin sık kullandığı sütunları ekler. Hem history hem plan_q1 için çalışır."""
    out = df.copy()
    period = pd.PeriodIndex(out.Month, freq="M")
    out["t"] = period.astype("int64")                              # ay sayacı, eğilim için
    out["month"] = period.month                                    # ayın numarası (1–12)
    price = out["Avg_Price_EUR"] if "Avg_Price_EUR" in out else out["Planned_Price_EUR"]
    # log(fiyat / liste fiyatı); 0 indirim olmadığını gösterir
    out["discount"] = np.log(price / out["List_Price_EUR"])
    past = history.set_index(["Product", "Month"]).Demand_Units
    a_year_earlier = [str(pd.Period(m, freq="M") - 12) for m in out.Month]
    # aynı ürünün on iki ay önceki talebi
    out["lag12"] = [past.get(key) for key in zip(out.Product, a_year_earlier)]
    return out


history, plan_q1 = add_features(history), add_features(plan_q1)

In [ ]:
# HAZIR · Çözücü: önce CBC denenir, açılmazsa HiGHS kullanılır

def _solver_works(solver):
    try:
        test = pulp.LpProblem("test", pulp.LpMinimize)
        x = pulp.LpVariable("x", cat="Binary")
        test += x
        test += x >= 1
        test.solve(solver)
        return pulp.LpStatus[test.status] == "Optimal"
    except Exception:
        return False


def _pick_solver():
    global pulp
    # gapRel=0 (CBC ve HiGHS): çözücü en iyi çözümü kanıtlayana kadar çalışır; varsayılan ayarla
    #   %0,01 farkla durabilir, bu modelde bu fark yarım saniyelik dengeyi gizleyebilir
    options = {"msg": False, "timeLimit": 60, "gapRel": 0}
    if _solver_works(pulp.PULP_CBC_CMD(msg=False)):
        return pulp.PULP_CBC_CMD(**options), "CBC"
    # yedek çözücü: yalnızca CBC açılmazsa (örneğin bazı Mac'lerde) kurulur
    if find_spec("highspy") is None:
        _install("highspy")
        _forget_pulp()
        import pulp
    if hasattr(pulp, "HiGHS") and _solver_works(pulp.HiGHS(msg=False)):
        # HiGHS'te ön çözümleme (presolve) kapalı: açıkken bu modelde en dengeli hattı zaman zaman
        #   kaçırıyor
        return pulp.HiGHS(**options, presolve="off"), "HiGHS"
    return None, None


SOLVER, SOLVER_NAME = _pick_solver()

if SOLVER:
    print(f"Veri yüklendi ✓ | çözücü: {SOLVER_NAME}")
else:
    print("Veri yüklendi, ama çözücü başlatılamadı. En kolay çözüm notebook'u Colab'de açmak. "
          "Kendi Mac'inizde çalışmak istiyorsanız Terminal'de 'pip install highspy' komutunu "
          "çalıştırıp çekirdeği yeniden başlatın.")
if _team():
    print(f"Takım: {_team()}")

In [ ]:
print("Talep geçmişi (son satırlar):")
show(history.tail(6), decimals=3)
print("Ocak–Mart 2026 planı:")
show(plan_q1, decimals=3)
demand = history.pivot(index="Month", columns="Product", values="Demand_Units")
demand.plot(figsize=(11, 3), title="Aylık talep (adet)")
plt.show()

## 1 · Talep tahmini (10–45. dakika)

**Amaç:** Ocak–Mart 2026 talebini ürün bazında, 31 Aralık 2025 itibarıyla tahmin etmek ve modelinize neden güvenilebileceğini geriye dönük bir testle göstermek.

**Elinizde neler var?**
- `history`: 2021–2025 aylık talep; promosyon, fiyat, iş günü sayısı ve takvim notuyla birlikte.
- `plan_q1`: Kerem'in gönderdiği Ocak–Mart 2026 planı. Promosyonların onay durumu `Promo_Status` sütununda.
- Hazır sütunlar: `t` (ay sayacı), `month` (ayın numarası), `discount` (fiyatın liste fiyatına oranının logaritması) ve `lag12` (aynı ürünün on iki ay önceki talebi).
- Hazır fonksiyonlar: `wape()`, `bias()` ve her modelin geçmesi gereken kıyas yöntemi `seasonal_naive()`. Bir tablonun fiyat ya da promosyon sütunlarını değiştirirseniz `add_features()` hazır sütunları yeniden hesaplar.

In [ ]:
# HAZIR · Tahmin araçları

def wape(actual, forecast):
    """WAPE: mutlak hataların toplamının gerçekleşen talep toplamına oranı. Düşük olan daha iyi."""
    actual, forecast = np.asarray(actual, float), np.asarray(forecast, float)
    return np.abs(actual - forecast).sum() / actual.sum()


def bias(actual, forecast):
    """Bias: (tahmin toplamı − gerçekleşen toplamı) / gerçekleşen toplamı. Pozitifse fazla,
    negatifse eksik tahmin ediyoruz."""
    actual, forecast = np.asarray(actual, float), np.asarray(forecast, float)
    return (forecast - actual).sum() / actual.sum()


def seasonal_naive(train, test):
    """Kıyas yöntemi: geçen yılın aynı ayının talebi."""
    return test["lag12"].to_numpy(float)


def check_backtest(backtest):
    """Geriye dönük testinizi kıyas yöntemiyle çalıştırır ve doğru sonuçla karşılaştırır."""
    problems = set()

    def probe(train, test):
        if len(train) and train.t.max() >= test.t.min():
            problems.add("future")
        if "Demand_Units" in test.columns:
            problems.add("actuals")
        return seasonal_naive(train, test)

    bt = backtest(probe)
    if len(bt) == 0:
        print("Geriye dönük test henüz bir şey döndürmüyor.")
    elif "future" in problems:
        print("✗ Bazı başlangıç aylarında model, tahmin ettiği aylardan sonraki verileri de "
              "görüyor.")
    elif "actuals" in problems:
        print("✗ test tablosunda hâlâ Demand_Units sütunu var, yani model gerçekleşen talebi "
              "görebiliyor. Modeli çağırmadan önce bu sütunu çıkarın: "
              "test.drop(columns='Demand_Units').")
    else:
        w, b = wape(bt.Demand_Units, bt.Forecast), bias(bt.Demand_Units, bt.Forecast)
        if len(bt) == 72 and abs(w - NAIVE_WAPE) < 0.0005 and abs(b - NAIVE_BIAS) < 0.0005:
            print(f"✓ Veriyi doğru ayırmışsınız: {len(bt)} tahmin, mevsimsel naif için WAPE "
                  f"{pct(w)}, bias {pct(b, signed=True)}.")
        else:
            print("✗ Doğru kurulmuş bir test, mevsimsel naif yöntem için 72 tahmin, WAPE "
                  f"{pct(NAIVE_WAPE)} ve bias {pct(NAIVE_BIAS, signed=True)} verir. Sizin "
                  f"sonucunuz: {len(bt)} tahmin, WAPE {pct(w)}, bias {pct(b, signed=True)}.")


def compare(models, backtest):
    """Verilen modellerin hepsini geriye dönük testten geçirir; WAPE, bias ve ürün bazında bias'ı
    gösterir."""
    rows = {}
    for name, model in models.items():
        bt = backtest(model)
        if len(bt) == 0:
            print("Geriye dönük test henüz bir şey döndürmüyor.")
            return
        w = wape(bt.Demand_Units, bt.Forecast)
        rows[name] = {"WAPE": pct(w), "Bias": pct(bias(bt.Demand_Units, bt.Forecast), signed=True)}
        for p, g in bt.groupby("Product"):
            rows[name][f"Bias {p}"] = pct(bias(g.Demand_Units, g.Forecast), signed=True)
        if w < 0.01:
            print(f"⚠ {name}: WAPE {pct(w)}. Bu kadar düşük bir hata genellikle modelin "
                  "gerçekleşen değerleri gördüğü anlamına gelir; veri sızıntısı olup olmadığını "
                  "kontrol edin.")
    display(pd.DataFrame.from_dict(rows, orient="index"))


def save_forecast(forecast, download=None):
    """Tahmin tablosunun biçimini kontrol eder, adetleri yuvarlar ve submission/forecast.csv
    dosyasını takım numarası ve kayıt saatiyle yazar. Colab'de TeamXX_forecast.csv adıyla
    indirir; tahmin hâlâ kıyas yönteminin tahminiyse download=True verilmedikçe indirmez."""
    columns = ["Month", "Product", "Forecast_Units"]
    if not set(columns) <= set(forecast.columns):
        print("Tahmin tablosunda Month, Product ve Forecast_Units sütunları olmalı; dosya "
              "kaydedilmedi.")
        return
    out = forecast[columns].copy()
    units = pd.to_numeric(out.Forecast_Units, errors="coerce")
    keys = set(zip(out.Month.astype(str), out.Product.astype(str)))
    expected = {(m, p) for m in MONTHS for p in PRODUCTS}
    if len(out) != 9 or keys != expected or units.isna().any() or (units <= 0).any():
        print("Tahmin tablosunda Ocak–Mart 2026'nın her ayı ve her ürün için tam bir satır "
              "(toplam 9 satır) bulunmalı ve bütün değerler pozitif olmalı; dosya kaydedilmedi.")
        return
    out["Forecast_Units"] = units.round().astype(int)
    team = _team()
    out["Team"] = team or ""
    saved_at = datetime.now(ZoneInfo("Europe/Istanbul")).isoformat(timespec="seconds")
    out["Saved_At"] = saved_at                       # dosyanın kaydedildiği zaman (İstanbul saati)
    out.to_csv(SUBMISSION / "forecast.csv", index=False)
    name = UPLOAD / f"{team or 'TeamXX'}_forecast.csv"
    shutil.copy(SUBMISSION / "forecast.csv", name)
    print(f"Kaydedildi: submission/forecast.csv (saat {saved_at[11:16]}). Puanlanan tahmin, 45. "
          "dakikaya kadar forma yüklediğiniz dosyadır.")
    baseline = "lag12" in forecast and (out.Forecast_Units == forecast.lag12.round()).all()
    if download is None:
        download = not baseline                      # kıyas yönteminin tahmini indirilmez
    if baseline and not download:
        print("Not: bu tahmin hâlâ kıyas yönteminin (mevsimsel naif) tahmini. Kendi modelinizi "
              "kurunca hücreyi yeniden çalıştırın; Colab'de dosya o zaman bilgisayarınıza iner.")
    if download and "google.colab" in sys.modules:
        from google.colab import files
        files.download(str(name))
        print(f"{name.name} bilgisayarınıza indi. 45. dakikaya kadar “45. dakika tahmini” formuna "
              "yükleyin ve formda takımınızı seçin. Form (eğitmeninizin ekranda gösterdiği): "
              "https://forms.gle/8aogMg4kcKzk1ipbA")


def plot_forecast(history, forecast):
    """Geçmiş talebi ve Ocak–Mart 2026 tahminini ürün bazında aynı grafikte gösterir (tahmin
    kesikli çizgi)."""
    past = history.pivot(index="Month", columns="Product", values="Demand_Units")
    ahead = forecast.pivot(index="Month", columns="Product", values="Forecast_Units")
    ahead = pd.concat([past.tail(1), ahead])         # kesikli çizgi aralık 2025'ten başlasın
    past.index = pd.PeriodIndex(past.index, freq="M")
    ahead.index = pd.PeriodIndex(ahead.index, freq="M")
    ax = past.plot(figsize=(11, 3.5), title="Aylık talep ve Ocak–Mart 2026 tahmini (adet)")
    colours = [line.get_color() for line in ax.get_lines()]
    ahead.plot(ax=ax, style="--", color=colours, legend=False)
    return ax

### Adım 1.1 · Geriye dönük test

Bu notebook'ta bir model, `(train, test)` tablolarını alıp `test` tablosunun her satırı için aynı sırayla bir tahmin döndüren bir fonksiyondur. En basit örneği `seasonal_naive`. Sizden istenen, her başlangıç ayı için `train` ve `test` tablolarını doğru seçmek ve model `test` tablosunu görmeden önce `Demand_Units` sütununu çıkarmak. Başlangıç ayları çalışma esaslarında verildi.

Fonksiyonu yazdıktan sonra hemen alttaki kontrol hücresini çalıştırın.

In [ ]:
# çalışma esaslarındaki 8 başlangıç ayı
ORIGINS = ["2023-12", "2024-03", "2024-06", "2024-09", "2024-12", "2025-03", "2025-06", "2025-09"]


def backtest(model):
    """Geriye dönük test. Her tahmin için bir satır döndürür: Month, Product, Demand_Units ve
    Forecast."""
    results = []
    # TODO ORIGINS listesindeki her başlangıç ayı için:
    #        train = model kurulurken kullanılabilecek satırlar
    #        test  = modelin tahmin edeceği satırlar (model bunları Demand_Units olmadan görür)
    #        forecast = model(train, test.drop(columns="Demand_Units"))
    #        test satırlarını Forecast sütunuyla birlikte results listesine ekleyin
    if not results:
        return pd.DataFrame(columns=["Month", "Product", "Demand_Units", "Forecast"])
    return pd.concat(results)

In [ ]:
check_backtest(backtest)

### Adım 1.2 · Kendi modeliniz

Aşağıda birkaç yön var; birini seçebilir, birleştirebilir ya da kendi yolunuzu bulabilirsiniz:

- **Üstel düzeltme** (örneğin `statsmodels` içindeki Holt-Winters): yalnızca talep serisine bakar ve seriden düzey, eğilim (trend) ve on iki aylık mevsimsellik bileşenlerini öğrenir.
- **log(talep) için regresyon:** verideki açıklayıcı değişkenleri kullanabilir. Bir talep planlamacısı bu değişkenlerden hangilerinin talebi etkilemesini bekler?
- **Ağaç tabanlı modeller** (`scikit-learn`): aynı değişkenleri kullanabilir. Ürün başına yalnızca 60 ay olduğu için modelin veriyi ezberleyip ezberlemediğine (aşırı öğrenme) dikkat edin. Ağaç modelleri eğilimi eğitim verisinin dışına taşıyamaz; önce eğilimi çıkarmayı düşünün.

Modelleri karşılaştırırken yalnızca toplam WAPE'ye değil, ürün bazında bias'a da bakın. Toplamda iyi görünen bir model, tek bir üründe sürekli aynı yönde yanılıyor olabilir.

In [ ]:
def my_model(train, test):
    """Sizin modeliniz. test tablosunun her satırı için, aynı sırayla bir tahmin döndürmeli."""
    # TODO modelinizi burada kurun; şimdilik kıyas yönteminin tahmini dönüyor
    return seasonal_naive(train, test)

In [ ]:
# denediğiniz diğer modelleri de buraya ekleyin
compare({"Mevsimsel naif": seasonal_naive, "Benim modelim": my_model}, backtest)

### Adım 1.3 · Ocak–Mart 2026 tahmini

Seçtiğiniz modelle üç ayı tahmin edin ve dosyayı kaydedin. Hücre dosyayı `TeamXX_forecast.csv` adıyla bilgisayarınıza indirir; bu dosyayı **45. dakikaya kadar** “45. dakika tahmini” formuna yükleyin. Puanlanan tahmin bu olacak; sonradan, örneğin konsensüs planı açıklandıktan sonra yapılan değişiklikler sayılmaz. Daha önce yüklediyseniz yeniden yükleyebilirsiniz: 45. dakikadan önceki son dosya sayılır.

Tahmin hâlâ kıyas yönteminin tahminiyse (yani `my_model` henüz değişmediyse) dosya inmez. O hâliyle indirmek isterseniz `save_forecast(forecast, download=True)` yazın. Alttaki hücre geçmişi ve tahmininizi aynı grafikte gösterir; tahmin ve plan slaytında kullanabilirsiniz.

In [ ]:
future = plan_q1.copy()
# TODO tahmin için kullanacağınız future tablosunu hazırlayın

forecast = future.assign(Forecast_Units=my_model(history, future))
save_forecast(forecast)
shown = ["Month", "Product", "Promo", "Planned_Price_EUR", "Forecast_Units"]
show(forecast[shown].round({"Forecast_Units": 0}))

In [ ]:
# geçmiş talep ve tahmininiz: tahmin ve plan slaytı için
plot_forecast(history, forecast)
plt.show()

## 2 · Üretim planı ve takt süresi (45–57. dakika)

**Amaç:** Konsensüs talep planını her ay için planlanan üretime ve takt süresine çevirmek, hattın hangi aya göre tasarlanacağına karar vermek.

**Elinizde neler var?** 45. dakikada duyurulan şifreyle açılan konsensüs talep planı ve Deniz'in e-postası; ayrıca `calendar` (iş günü, vardiya, mola, bakım ve ek duruşlar) ve `products` (31 Aralık stoku ve ilk geçiş verimi). Bundan sonra herkes konsensüs talep planıyla çalışıyor.

In [ ]:
# HAZIR · Plan araçları

def unlock_release(file, password):
    """Şifreli dosyayı notebook'un yanındaki release/ klasörüne açar."""
    if not password:
        print("Salonda duyurulan şifreyi yazın ve hücreyi yeniden çalıştırın.")
        return
    # büyük harf ya da boşlukla yazılsa da çalışır
    password = "-".join(password.lower().split())
    with zipfile.ZipFile(file) as z:
        for member in z.infolist():
            try:
                z.extract(member, ".", pwd=password.encode())
            except RuntimeError:
                print("Şifre yanlış, lütfen tekrar deneyin.")
                return
            except (zipfile.BadZipFile, zlib.error):
                Path(member.filename).unlink(missing_ok=True)       # yarım kalan dosya silinir
                print("Şifre yanlış, lütfen tekrar deneyin.")
                return
        print("Açılan dosyalar:", ", ".join(n for n in z.namelist() if not n.endswith("/")))


BACKUP_FILES = ("consensus_demand.csv", "Deniz_Email_45_min.pdf",
                "urgent_order.csv", "Urgent_Email_95_min.pdf")


def use_backup_release():
    """Şifreli dosya açılmazsa: eğitmenin verdiği şifresiz dosyaları (Colab'e yüklenmiş ya da
    notebook'un yanında) release/ klasörüne taşır."""
    RELEASE.mkdir(exist_ok=True)
    moved = []
    for folder in (Path.cwd(), Path.cwd().parent, CONTENT, CONTENT / "drive" / "MyDrive"):
        for file in BACKUP_FILES:
            if (folder / file).exists() and not (RELEASE / file).exists():
                shutil.move(str(folder / file), str(RELEASE / file))
                moved.append(file)
    print("release/ klasörüne taşınan dosyalar:", ", ".join(moved) if moved else "—")


def load_consensus():
    """Konsensüs talep planını {(ay, ürün): adet} biçiminde bir sözlük olarak döndürür."""
    if not (RELEASE / "consensus_demand.csv").exists():
        print("Önce 45. dakika dosyasını açın.")
        return {}
    demand = pd.read_csv(RELEASE / "consensus_demand.csv")
    show(demand.pivot(index="Month", columns="Product", values="Consensus_Demand_Units"))
    return {(r.Month, r.Product): r.Consensus_Demand_Units for r in demand.itertuples()}


def add_urgent(D):
    """ACİL siparişi konsensüs talebine (D) ekler ve yeni sözlüğü döndürür."""
    if not D:
        print("Önce 45. dakika dosyasını açın.")
        return {}
    if not (RELEASE / "urgent_order.csv").exists():
        print("Önce 95. dakika dosyasını açın.")
        return dict(D)
    urgent = pd.read_csv(RELEASE / "urgent_order.csv")
    show(urgent)
    out = dict(D)
    for r in urgent.itertuples():
        out[(r.Month, r.Product)] += r.Additional_Units
    return out


def _mark(ok):
    return "✓" if ok else "✗"


INTRO_MIX = [0.4, 0.35, 0.25]                        # girişte önerilen geçici ürün karması


def _intro_mix(mix):
    """Ürün karması hâlâ girişteki geçici değerse True döndürür."""
    return bool(mix) and [mix.get(p) for p in PRODUCTS] == INTRO_MIX


def check_plan(avail, plan, takt=None, design_month=None, C=None, mix=None):
    """Ocak sonuçlarınızı doğru değerlerle karşılaştırır; DESIGN_MONTH, C ve MIX'in birbiriyle
    tutarlı olup olmadığına bakar."""
    avail, takt, mix = _as_dict(avail), _as_dict(takt) or {}, _as_dict(mix)
    if not avail or plan is None or len(plan) == 0:
        print("available_time() ve production_plan() henüz bir sonuç döndürmüyor.")
        return
    jan_planned = plan[plan.Month == "2026-01"].Planned_Production.sum()
    ok = abs(avail.get("2026-01", 0) - JAN_AVAILABLE) < 1
    print(_mark(ok), f"Ocak ayının kullanılabilir süresi (doğru değer: {fmt(JAN_AVAILABLE)} sn)")
    ok = abs(jan_planned - JAN_PLANNED) <= 3
    print(_mark(ok), f"Ocak ayının toplam planlanan üretimi (doğru değer: {fmt(JAN_PLANNED)} adet)")
    if takt:
        ok = abs(takt.get("2026-01", 0) - JAN_TAKT) < 0.03
        print(_mark(ok), f"Ocak ayının takt süresi (doğru değer: {fmt(JAN_TAKT, 2)} sn)")
    if C is not None or mix:
        design_takt = takt.get(design_month)
        ok = (bool(mix) and abs(sum(mix.values()) - 1) < 1e-6
              and C is not None and design_takt is not None and abs(C - design_takt) < 0.01)
        print(_mark(ok), "MIX'teki payların toplamı 1 ve C, DESIGN_MONTH ayının takt süresine "
                         "eşit (bu kontrol hangi ayı seçtiğinize bakmaz)")
    if _intro_mix(mix):
        print("⚠ MIX hâlâ girişte önerilen geçici değer. Bölüm 2'de hesapladığınız ürün karmasını "
              "yazın.")

### Adım 2.1 · Dosyayı açın

Salonda duyurulan şifreyi aşağıdaki hücreye yazıp çalıştırın. Ardından Deniz'in e-postasını (`release/Deniz_Email_45_min.pdf`) okuyun.

Şifreli dosya bir türlü açılmazsa eğitmeniniz size dosyaların şifresiz kopyalarını verir: onları Colab'in sol kenarındaki *Dosyalar* paneline sürükleyip bırakın ve `use_backup_release()` hücresini çalıştırın; dosyalar `release/` klasörüne taşınır.

In [ ]:
unlock_release("release_45_min.zip", password="")   # ← şifreyi buraya yazın
D = load_consensus()                                # {(ay, ürün): konsensüs talep}

### Adım 2.2 · Kullanılabilir süre ve planlanan üretim

Kullanılabilir sürenin formülü çalışma esaslarında. Planlanan üretimin nasıl hesaplanacağını Deniz e-postasında anlatıyor; hesaplama notlarını da göz önünde bulundurun.

In [ ]:
def available_time(calendar):
    """Her ayın kullanılabilir süresini saniye cinsinden döndürür: {ay: saniye}."""
    # TODO çalışma esaslarındaki kullanılabilir süre formülü
    return {}


def production_plan(D):
    """Her ay ve ürün için bir satır döndürür: Month, Product ve Planned_Production."""
    # TODO Deniz'in e-postasındaki tarif ve hesaplama notları
    return pd.DataFrame(columns=["Month", "Product", "Planned_Production"])


avail, plan = available_time(calendar), production_plan(D)

### Adım 2.3 · Takt süresi ve tasarım ayı

Her ayın takt süresini hesaplayın ve hattı hangi aya göre tasarlayacağınıza karar verin. Bu ayı `DESIGN_MONTH`'a, o ayın takt süresini `C`'ye, ürün karmasını da `MIX`'e yazın. Ürün karmasının tanımı çalışma esaslarında.

Ardından kontrol hücresini çalıştırın. Kontrol, ocak için doğru değerleri gösterir; diğer ayları doğru hesaplamak size kalıyor. `DESIGN_MONTH`, `C` ve `MIX` için yalnızca birbirleriyle tutarlı olup olmadıklarına bakar; doğru ayı seçip seçmediğinizi söylemez.

In [ ]:
takt = {}                                     # {ay: saniye cinsinden takt süresi}
# tasarım ayı, o ayın takt süresi ve ürün karması {ürün: pay}
DESIGN_MONTH, C, MIX = None, None, None
# TODO her ayın takt süresini hesaplayın; sonra DESIGN_MONTH, C ve MIX değerlerini belirleyin

In [ ]:
check_plan(avail, plan, takt, DESIGN_MONTH, C, MIX)

## 3 · Hat dengeleme (57–90. dakika)

**Amaç:** Üç ay boyunca değişmeden çalışabilecek bir hat tasarlamak. Ne beklendiğini Selin'in e-postası anlatıyor; hattın uyması gereken kurallar ise Hasan'ın ve Selin'in mesajlarında.

**Elinizde neler var?**
- `TASKS` (işler), `TIME[iş][ürün]` (saniye cinsinden süreler), `GROUP[iş]` (süreç grubu), `ARCS` (öncelik ilişkileri, yani önce–sonra çiftleri) ve `CURRENT_LINE` (bugünkü hat). İşlerin adları `tasks` tablosunda.
- `weighted(mix)`: ürün karmasına göre ağırlıklı iş süreleri. `station_loads(line, mix)`: her istasyonun işleri, ağırlıklı yükü, ürün bazında süresi ve süreç grupları; `station_table(line, mix)` aynı bilgiyi tablo olarak gösterir. `capacity(line, mix, seconds)`: bir hattın verilen sürede kaç adet üretebileceği.
- `check_assignment(line, C, mix)`: hakem. Yalnızca temel kurallara bakar: her iş tam olarak bir istasyonda mı, iş sırası korunuyor mu, istasyon yükleri takt süresi içinde mi, istasyonlar 1'den başlayıp boşluksuz mu numaralanmış. Yazışmalardaki kurallar sizin işiniz.

In [ ]:
# HAZIR · Hat araçları

def weighted(mix):
    """Verilen ürün karması {ürün: pay} için ağırlıklı iş sürelerini döndürür."""
    mix = dict(mix)
    return {i: sum(mix[p] * TIME[i][p] for p in PRODUCTS) for i in TASKS}


def station_loads(line, mix):
    """Her istasyonun işleri, ağırlıklı yükü, ürün bazında süresi ve süreç grupları; yuvarlanmamış
    değerlerle."""
    w, out = weighted(mix), {}
    for task, k in sorted(dict(line).items()):
        if task not in TIME:
            continue
        if k not in out:
            out[k] = {"Tasks": [], "Load": 0.0, "Groups": set(), **{p: 0 for p in PRODUCTS}}
        s = out[k]
        s["Tasks"].append(task)
        s["Load"] += w[task]
        s["Groups"].add(GROUP[task])
        for p in PRODUCTS:
            s[p] += TIME[task][p]
    return dict(sorted(out.items()))


def station_table(line, mix):
    """Her istasyon için bir satır: işler, ağırlıklı yük, ürün bazında süre ve süreç grupları
    (yükler gösterim için yuvarlanır)."""
    rows = {k: {"Tasks": ", ".join(s["Tasks"]), "Weighted": round(s["Load"], 2),
                **{p: s[p] for p in PRODUCTS}, "Groups": ", ".join(sorted(s["Groups"]))}
            for k, s in station_loads(line, mix).items()}
    return pd.DataFrame.from_dict(rows, orient="index")


def check_assignment(line, C, mix):
    """Hakem: verilen hatta ({iş: istasyon}) yalnızca temel kuralları denetler: her iş bir
    istasyonda, iş sırası, takt süresi ve istasyon numaraları. Boş liste, bu kurallarda sorun
    olmadığı anlamına gelir; yazışmalardaki kurallara bakmaz."""
    line = dict(line)
    issues = [f"{i} hiçbir istasyona atanmamış." for i in TASKS if i not in line]
    issues += [f"{i} diye bir iş yok." for i in line if i not in TIME]
    issues += [f"İş sırası bozuk: {a} ({line[a]}. istasyon), {b} işinden ({line[b]}. istasyon) "
               "önce gelmeli."
               for a, b in ARCS if a in line and b in line and line[a] > line[b]]
    stations = sorted(set(line.values()))
    if stations != list(range(1, len(stations) + 1)):
        issues.append("İstasyonlar 1'den başlayıp boşluksuz numaralanmalı (şu anki numaralar: "
                      f"{stations}).")
    for k, s in station_loads(line, mix).items():
        if s["Load"] > C + 1e-9:
            issues.append(f"{k}. istasyon: {fmt(s['Load'], 2)} sn > takt {fmt(C, 2)} sn")
    return issues


def capacity(line, mix, seconds):
    """Hattın verilen sürede (saniye) kaç adet üretebileceğini döndürür."""
    loads = station_loads(line, mix)
    if not loads:
        return 0
    busiest = max(s["Load"] for s in loads.values())
    return math.floor(seconds / busiest)


def _own_rules(rules, line, C, mix):
    """Sizin kural fonksiyonunuzu çalıştırır ve sonucu bir liste olarak döndürür."""
    found = rules(line, C, mix)
    if found is None:
        print("Not: check_rules bir liste döndürmedi (None), bu yüzden kurallarınız denetlenmedi. "
              "Fonksiyonun sonunda return problems satırı olmalı.")
    return list(found or [])


def _report(line, C, mix, rules=None):
    issues = check_assignment(line, C, mix)
    print("Hakem (temel kurallar):" + ("" if issues else " sorun yok"))
    for issue in issues:
        print("  -", issue)
    if rules is None:
        print("Hakem yazışmalardaki kurallara bakmaz: rules=check_rules verin.")
        return
    problems = _own_rules(rules, line, C, mix)
    print("Sizin kurallarınız (check_rules):" + ("" if problems else " sorun bulunmadı"))
    for problem in problems:
        print("  -", problem)


def show_current_line(C, mix, rules=None):
    """Mevcut hattı gösterir; hakeme ve verilirse sizin kurallarınıza (rules) sorar."""
    mix = _as_dict(mix)
    if C is None or not mix:
        print("Önce Bölüm 2'de C ve MIX değerlerini belirleyin ya da şimdilik geçici değerlerle "
              "deneyin: C, MIX = 85, {'ND100': 0.4, 'ND200': 0.35, 'ND300': 0.25}")
        return
    print("Mevcut hat, sizin takt süreniz ve ürün karmanızla:")
    show(station_table(CURRENT_LINE, mix))
    _report(CURRENT_LINE, C, mix, rules)


def report_line(line, C, mix, plan, takt, bound=None, rules=None):
    """İstasyonları gösterir; hakeme ve verilirse sizin kurallarınıza (rules) sorar; hattı üç ayın
    her birinde sınar."""
    mix, takt = _as_dict(mix), _as_dict(takt)
    if C is None or not mix:
        print("Önce Bölüm 2'de C ve MIX değerlerini belirleyin ya da şimdilik geçici değerlerle "
              "deneyin: C, MIX = 85, {'ND100': 0.4, 'ND200': 0.35, 'ND300': 0.25}")
        return
    if line is None:
        print("Model uygun bir hat bulamadı (balance_line None döndürdü). Kısıtlarınızı ve C "
              "değerini kontrol edin.")
        return
    line = dict(line)
    if not line:
        print("Henüz bir hat yok: balance_line boş bir sözlük döndürüyor.")
        return
    table = station_table(line, mix)
    show(table)
    lower = bound(C, mix) if bound else None
    print(f"İstasyon: {len(table)} | alt sınır: {'—' if lower is None else lower} | "
          f"en yoğun istasyon: {fmt(table.Weighted.max(), 2)} sn | takt: {fmt(C, 2)} sn")
    _report(line, C, mix, rules)
    if plan is not None and len(plan) and takt and all(m in takt for m in MONTHS):
        print("Üç ay, her biri kendi takt süresi ve ürün karmasıyla:")
        for m in MONTHS:
            units = plan[plan.Month == m].set_index("Product").Planned_Production
            mix_m = (units / units.sum()).to_dict()
            issues = check_assignment(line, takt[m], mix_m)
            if rules is not None:
                issues += _own_rules(rules, line, takt[m], mix_m)
            lower = bound(takt[m], mix_m) if bound else None
            shown = "" if lower is None else f"alt sınır {lower} | "
            print(" ", m, shown + ("sorun yok" if not issues else "; ".join(issues)))

### Adım 3.1 · Hattın kuralları ve bugünkü hat

Selin ve Hasan'ın mesajlarında hat için kurallar var; bunları bulun, hem modelinize hem kendi kontrolünüze yazın. Kontrolünüz `check_rules(line, C, mix)`: bir hattı alır ve bulduğu sorunları bir liste olarak döndürür; sorun yoksa liste boş kalır.

Kontrolünüzü önce bugünkü hatta deneyin: sizin bulduğunuz takt süresi ve ürün karmasıyla hangi istasyon, neden sorun çıkarıyor?

In [ ]:
def check_rules(line, C, mix):
    """Yazışmalarda bulduğunuz hat kuralları. Kuralı çiğneyen her durum için problems listesine
    kısa bir açıklama ekleyin; boş liste, sorun bulunmadığı anlamına gelir."""
    problems = []
    # TODO yazışmalarda bulduğunuz her kural için bir kontrol yazın
    #      (station_loads(line, mix): her istasyonun işleri, yükü, ürün süreleri ve grupları)
    return problems

In [ ]:
show_current_line(C, MIX, rules=check_rules)

### Adım 3.2 · Alt sınır ve optimizasyon modeli

Önce `lower_bound` fonksiyonuyla istasyon sayısının alt sınırını hesaplayın (formülü hazırlık notunda). Ardından `balance_line` fonksiyonunun içine modelinizi kurun: karar değişkenleri, amaç fonksiyonu ve kısıtlar. Fonksiyon, hattı `{iş: istasyon}` biçiminde bir sözlük olarak döndürmeli; istasyon numaraları 1'den başlar. Çözücü hazır, `model.solve(SOLVER)` ile çağırabilirsiniz.

Çalışma esaslarında istendiği gibi model **doğrusal** olmalı ve **tek bir amaç fonksiyonu** kullanmalı.

Bir sonraki hücre hattınızı gösterir, hakeme ve sizin `check_rules` fonksiyonunuza sorar, aynı düzeni üç ayın her birinde sınar ve her ay için, o ayın takt süresi ve karmasıyla `lower_bound` sonucunu da yazar (tahmin ve plan slaytındaki tablo için). Tek bir ayda bile sorun çıkıyorsa tasarımınızı gözden geçirin.

In [ ]:
def lower_bound(C, mix):
    """İstasyon sayısının alt sınırı (formülü hazırlık notunda)."""
    # TODO
    return None


def balance_line(C, mix):
    """Hat modelinizi kurar ve çözer. {iş: istasyon} sözlüğü döndürür; uygun bir hat yoksa None."""
    w = weighted(mix)                                  # ürün karmasına göre ağırlıklı iş süreleri
    model = pulp.LpProblem("line", pulp.LpMinimize)

    # TODO karar değişkenleri

    # TODO amaç fonksiyonu (tek bir amaç fonksiyonu)

    # TODO kısıtlar: hattın uyması gereken bütün kurallar (yazışmaları yeniden okuyun)

    model.solve(SOLVER)
    if pulp.LpStatus[model.status] != "Optimal":
        return None
    # TODO her işin hangi istasyona atandığını değişkenlerinizden okuyun
    return {}

In [ ]:
new_line = balance_line(C, MIX) if C is not None and MIX is not None else None
report_line(new_line, C, MIX, plan, takt, bound=lower_bound, rules=check_rules)

### Adım 3.3 · Model kurulamazsa: yedek yol

`heuristic_line(C, MIX)` sizin için hazır bir hat kurar. Kullandığı yöntemin adı *sıralı konum ağırlığı* (ranked positional weight): her işin konum ağırlığı, kendi süresiyle ondan sonra gelmesi gereken bütün işlerin sürelerinin toplamıdır. Yöntem istasyonları tek tek doldurur ve her seferinde, öncülleri atanmış ve istasyona sığan işler arasından konum ağırlığı en büyük olanı seçer.

Bu yöntem yalnızca iş sırasına ve takt süresine bakar; yazışmalardaki kuralları bilmez. Bulduğu hattı `check_rules` ile denetleyin. İsterseniz HAZIR hücredeki `heuristic_line` fonksiyonunu, içindeki `fits` fonksiyonuyla birlikte kendi hücrenize kopyalayıp kurallarınızı `fits`'e ekleyebilirsiniz. En az istasyonu ya da en dengeli hattı bulacağının da garantisi yoktur: bunu kesin olarak ancak bir optimizasyon modeli söyleyebilir.

In [ ]:
# HAZIR · Yedek yol: sezgisel yöntem

def heuristic_line(C, mix):
    """Sıralı konum ağırlığı yöntemiyle bir hat kurar. Yalnızca iş sırasına ve takt süresine bakar;
    en iyi hattı garanti etmez."""
    w = weighted(mix)
    successors = {i: [b for a, b in ARCS if a == i] for i in TASKS}
    predecessors = {i: [a for a, b in ARCS if b == i] for i in TASKS}

    def all_successors(i):
        found, stack = set(), list(successors[i])
        while stack:
            j = stack.pop()
            if j not in found:
                found.add(j)
                stack += successors[j]
        return found

    pos_weight = {i: w[i] + sum(w[j] for j in all_successors(i)) for i in TASKS}
    # atamalar {iş: istasyon}, doldurulan istasyonun numarası ve o istasyondaki işler
    line, k, here = {}, 1, []

    def fits(i, here):
        """i işi, doldurulmakta olan istasyona (here) eklenebilir mi?"""
        load = sum(w[j] for j in here + [i])
        return all(a in line for a in predecessors[i]) and load <= C + 1e-9

    while len(line) < len(TASKS):
        candidates = [i for i in TASKS if i not in line and fits(i, here)]
        if candidates:
            i = max(candidates, key=pos_weight.get)
            line[i] = k
            here.append(i)
        elif not here:                                # boş bir istasyon bile bu işi alamıyor
            raise ValueError("Bazı işler hiçbir istasyona sığmıyor.")
        else:                                         # hiçbir iş sığmıyor: yeni bir istasyon aç
            k, here = k + 1, []
    return line


# Kullanmak için aşağıdaki iki satırın başındaki # işaretini silin:
# new_line = heuristic_line(C, MIX)
# report_line(new_line, C, MIX, plan, takt, bound=lower_bound, rules=check_rules)

## 4 · ACİL sipariş (95–115. dakika)

**Amaç:** Ek sipariş için maliyetiyle birlikte bir öneri hazırlamak ve hangi durumda bu önerinin değişeceğini söylemek.

İkinci şifre `urgent_order.csv` dosyasını ve Ece'nin e-postasını açar; hücre siparişi talebe ekler (`D_URGENT`).

**Sizden istenen**
1. Siparişle birlikte yeni takt süresi, ürün karması ve alt sınır; siparişin kaç istasyon gerektirdiği.
2. Önden üretim seçeneği: ocakta kaç adet önden üretmek gerekir (`PREBUILD_UNITS`) ve ocak bunu kaldırabilir mi? `capacity(line, mix, seconds)` işinize yarayabilir.
3. İki seçeneğin maliyeti (Ece'nin tablosundan) ve her birinin bıraktığı kalan kapasite: ACİL slaytı için.
4. Ne önerdiğiniz ve hangi durumda fikrinizi değiştireceğiniz: Ece'nin sorusu, ACİL slaytının alt kutusu.

Hangi seçeneği önerirseniz önerin, kendi hattınızda işlemeli: istasyon eklerseniz yeni hat, yeni takt süresi ve karmayla hem hakemden hem kendi kurallarınızdan geçmeli; önden üretirseniz ocak bunu kaldırabilmeli.

In [ ]:
unlock_release("release_95_min.zip", password="")   # ← şifreyi buraya yazın
D_URGENT = add_urgent(D)                            # {(ay, ürün): sipariş dâhil talep}

In [ ]:
C_URGENT, MIX_URGENT, urgent_line = None, None, None
PREBUILD_UNITS = None
STATION_COST_Q1_EUR, PREBUILD_COST_EUR = None, None
STATION_SPARE_UNITS, PREBUILD_SPARE_UNITS = None, None
# TODO yeni plan, takt süresi (C_URGENT), karma (MIX_URGENT), alt sınır ve gereken hat (urgent_line)
# TODO önden üretim: adedi PREBUILD_UNITS'e yazın
# TODO maliyetler (Ece'nin tablosu): ek istasyonun çeyrek boyunca ve önden üretimin toplam maliyeti
# TODO ACİL slaytı için kalan kapasite (capacity işinize yarar): ek istasyonda yeni hat şubatta,
#      önden üretimde hattınız ocakta, gerekenin üstünde kaç adet daha üretebilir?

## 5 · Teslim (115. dakikaya kadar)

Cevaplarınızı aşağıdaki hücreye yazıp çalıştırın; hattınız hazır olunca, 90. dakika civarında bir kez çalıştırmanız iyi olur. Hücre `answers.csv` dosyasını ve atama dosyalarını (`assignment_main.csv` ve istasyon eklediyseniz `assignment_urgent.csv`) `submission/` klasörüne yazar; eksik bir cevap varsa listeler. Colab'de bütün cevaplar doluysa iki dosya bilgisayarınıza iner: `TeamXX_submission.zip` ve notebook'unuzun kendisi, `TeamXX_notebook.ipynb` (tarayıcı birden çok dosya indirmek için izin isterse izin verin). Eksik cevapla indirmek isterseniz son satıra `download=True` ekleyin.

**115. dakikaya kadar** “Final teslim” formuna üç dosya yükleyin: `TeamXX_submission.zip`, `TeamXX_sunum.pptx` (sunum şablonunu bu adla kaydedin) ve `TeamXX_notebook.ipynb`. Notebook inmezse *Dosya → İndir → .ipynb dosyasını indir* ile indirip adını değiştirin. Yeniden yüklerseniz son yüklediğiniz dosyalar sayılır; 115. dakikadan sonra gelen dosyalar kabul edilmez.

In [ ]:
# HAZIR · Teslim aracı

DECISIONS = ("add_stations", "prebuild")


def _write_line(line, file):
    table = pd.DataFrame(sorted(dict(line).items()), columns=["Task", "Station"])
    table.to_csv(SUBMISSION / file, index=False)


def save_submission(answers, line, urgent_line=None, download=None):
    """Cevapları ve atamaları submission/ klasörüne yazar, eksik cevapları listeler. Colab'de,
    bütün cevaplar doluysa ya da download=True verildiyse TeamXX_submission.zip ve
    TeamXX_notebook.ipynb dosyalarını indirir."""
    answers, line, urgent_line = dict(answers), _as_dict(line), _as_dict(urgent_line)
    decision = answers.get("urgent_decision")
    if isinstance(decision, str):
        decision = answers["urgent_decision"] = decision.strip().lower()
    if decision is not None and decision not in DECISIONS:
        print("urgent_decision yalnızca 'add_stations' ya da 'prebuild' olabilir (yazdığınız: "
              f"{decision!r}); bu cevap boş olarak kaydedildi.")
        answers["urgent_decision"] = decision = None
    team = _team()
    stamp = datetime.now(ZoneInfo("Europe/Istanbul")).isoformat(timespec="seconds")
    # takım numarası ve kayıt saati de cevap dosyasına yazılır
    rows = {**answers, "team": team or "", "saved_at": stamp}
    pd.DataFrame({"Key": list(rows), "Value": list(rows.values())}).to_csv(
        SUBMISSION / "answers.csv", index=False)
    for old in SUBMISSION.glob("assignment_*.csv"):  # önceki kayıttan kalan atama dosyaları silinir
        old.unlink()
    if line:
        _write_line(line, "assignment_main.csv")
    if decision == "add_stations":
        if urgent_line:
            _write_line(urgent_line, "assignment_urgent.csv")
        else:
            print("urgent_decision 'add_stations', ama urgent_line boş; bu yüzden "
                  "assignment_urgent.csv yazılmadı.")
    not_needed = {"prebuild_units"} if decision == "add_stations" else set()
    missing = [k for k, v in answers.items() if v is None and k not in not_needed]
    if _intro_mix({p: answers.get(f"mix_{p.lower()}") for p in PRODUCTS}):
        print("⚠ MIX hâlâ girişte önerilen geçici değer. Bölüm 2'de hesapladığınız ürün karmasını "
              "yazın.")
    print("submission/ klasöründeki dosyalar:", sorted(p.name for p in SUBMISSION.iterdir()))
    print("Eksik cevaplar:", missing if missing else "—")
    if download is None:
        download = not missing                       # eksik cevap varken indirmez
    if "google.colab" not in sys.modules:
        return
    if not download:
        print("Eksik cevaplar olduğu için teslim dosyası indirilmedi. Yine de indirmek için: "
              "save_submission(answers, new_line, urgent_line, download=True)")
        return
    from google.colab import files
    name = f"{team or 'TeamXX'}_submission"
    shutil.make_archive(str(UPLOAD / name), "zip", SUBMISSION)
    files.download(str(UPLOAD / f"{name}.zip"))
    print(f"{name}.zip bilgisayarınıza indi.")
    _download_notebook(team)
    print(f"Şimdi 115. dakikaya kadar “Final teslim” formuna üç dosya yükleyin: {name}.zip, "
          f"{team or 'TeamXX'}_sunum.pptx ve {team or 'TeamXX'}_notebook.ipynb. Form "
          "(eğitmeninizin ekranda gösterdiği): https://forms.gle/zkj9uEyNLHyWTqFs9 · Yeniden "
          "yüklerseniz son yüklediğiniz dosyalar sayılır.")


def _download_notebook(team):
    """Colab'de notebook'un kendisini TeamXX_notebook.ipynb adıyla indirir; olmazsa elle nasıl
    indirileceğini söyler."""
    path = UPLOAD / f"{team or 'TeamXX'}_notebook.ipynb"
    try:
        from google.colab import _message, files
        notebook = _message.blocking_request("get_ipynb", timeout_sec=30)["ipynb"]
        path.write_text(json.dumps(notebook), encoding="utf-8")
        files.download(str(path))
        print(f"{path.name} bilgisayarınıza indi.")
    except Exception:
        print("Notebook kendiliğinden inmedi: Dosya → İndir → .ipynb dosyasını indir ile indirip "
              f"adını {path.name} yapın.")

In [ ]:
# hesapladığınız değişkenler kendiliğinden gelir; None kalanları siz doldurun
mix = dict(MIX) if MIX is not None else {}
answers = {
    "takt_jan": takt.get("2026-01"), "takt_feb": takt.get("2026-02"),
    "takt_mar": takt.get("2026-03"),
    "design_month": DESIGN_MONTH,                # hattı tasarladığınız ay (YYYY-AA)
    "planned_production_design": None,           # tasarım ayında toplam planlanan üretim (Adım 2.2)
    "mix_nd100": mix.get("ND100"), "mix_nd200": mix.get("ND200"), "mix_nd300": mix.get("ND300"),
    # tasarım ayının takt süresiyle alt sınır (Adım 3.2)
    "lower_bound_design": None,
    "stations": None,                            # önerdiğiniz istasyon sayısı (Adım 3.2)
    "current_bottleneck_load": None,             # mevcut hatta en yoğun istasyonun yükü (Adım 3.1)
    "urgent_takt": C_URGENT,
    "urgent_lower_bound": None,                  # ACİL takt ve karmayla alt sınır (Bölüm 4)
    "urgent_decision": None,                     # "add_stations" ya da "prebuild"
    # ocakta önden üretilecek adet (add_stations seçtiyseniz boş kalabilir)
    "prebuild_units": PREBUILD_UNITS,
}
# TODO None olan değerleri doldurun

save_submission(answers, new_line, urgent_line)